In [1]:
import pandas as pd
import sqlite3

In [2]:
DB_NAME=r"C:\Users\RADHAGOPINATH\recovery_revenue.db"
conn=sqlite3.connect(DB_NAME)
cursor=conn.cursor()

In [7]:
cursor.execute("select * from customer")
rows=cursor.fetchall()

In [8]:
df=pd.DataFrame(rows,columns=["customer_id",
                              "prev_purchases",
                              "total_spending",
                              "clv",
                              "customer_status"])

In [9]:
df.head()

,customer_id,prev_purchases,total_spending,clv,customer_status
0,1,3,4281.73,5131.46,1
1,2,11,6037.09,7842.49,1
2,3,11,52174.07,61098.38,1
3,4,10,28131.15,34821.97,1
4,5,3,3194.69,4649.28,1


In [30]:
cursor.execute("select * from transactions")
trows=cursor.fetchall()

In [31]:
t_df=pd.DataFrame(trows,columns=["transaction_id","amount","payment_status","customer_id","event_type","failure_reason","cart_value","items_count","timestamp"])

In [32]:
t_df.head()

,transaction_id,amount,payment_status,customer_id,event_type,failure_reason,cart_value,items_count,timestamp
0,1,8131.83,SUCCESS,226,SUCCESSFUL_PURCHASE,None,8131.83,3,2026-06-01 17:55:55
1,2,6877.79,SUCCESS,522,SUCCESSFUL_PURCHASE,None,6877.79,4,2026-06-07 06:44:55
2,3,17608.35,SUCCESS,99,SUCCESSFUL_PURCHASE,None,17608.35,4,2026-07-01 02:47:56
3,4,18723.62,SUCCESS,959,SUCCESSFUL_PURCHASE,None,18723.62,7,2026-07-30 23:40:56
4,5,8110.44,SUCCESS,506,SUCCESSFUL_PURCHASE,None,8110.44,8,2026-07-17 18:29:52


In [36]:
t_df[(t_df["customer_id"]==3)]

,transaction_id,amount,payment_status,customer_id,event_type,failure_reason,cart_value,items_count,timestamp
814,815,8802.78,SUCCESS,3,SUCCESSFUL_PURCHASE,None,8802.78,8,2026-08-15 21:25:23
1910,1911,0.00,NOT_COMPLETED,3,CHECKOUT_ABANDONED,None,14557.92,1,2026-06-09 03:18:36
3657,3658,5830.20,FAILED,3,PAYMENT_FAILED,EXPIRED_CARD,5830.20,2,2026-06-17 02:40:05


In [39]:
cust_transaction=t_df[(t_df["customer_id"]==3) & (t_df["transaction_id"]==3658)]
transaction_id=cust_transaction.iloc[0]["transaction_id"]
customer_id=cust_transaction.iloc[0]["customer_id"]
event_type=cust_transaction.iloc[0]["event_type"]
amount=cust_transaction.iloc[0]["amount"]
cart_value=cust_transaction.iloc[0]["cart_value"]
timestamp=cust_transaction.iloc[0]["timestamp"]

In [40]:
if event_type =="PAYMENT_FAILED":
    current_transaction_value = amount
elif event_type=="CHECKOUT_ABANDONED":
    current_transaction_value = cart_value
elif event_type=="SUCCESSFUL_PURCHASE":
    current_transaction_value = amount

In [45]:
customer=df[df["customer_id"]==3]
total_spending=customer.iloc[0]["total_spending"]
clv=customer.iloc[0]["clv"]
prev_purchases=customer.iloc[0]["prev_purchases"]
if prev_purchases>0:
    avg_order_value=total_spending/prev_purchases
    print(avg_order_value)
    print("Returning")
else:
    avg_order_value=0
    print("New customer")

if clv < 9_000:
    customer_value="LOW"
elif clv < 70_000:
    customer_value="MEDIUM"
else:
    customer_value="HIGH"
print(customer_value)

4743.097272727273
Returning
MEDIUM


In [41]:
df["clv"].describe()

count      1600.000000
mean      52394.332906
std       65838.370576
min         200.210000
25%        8695.230000
50%       26335.925000
75%       68714.050000
max      427006.670000
Name: clv, dtype: float64

In [48]:
def analyze_customer(customer_id, transaction_id):
    result={}
    # first get the customers details
    customer=df[df["customer_id"]==customer_id]
    total_spending=customer.iloc[0]["total_spending"]
    clv=customer.iloc[0]["clv"]
    prev_purchases=customer.iloc[0]["prev_purchases"]

    #now get the transactions details
    cust_transaction=t_df[(t_df["customer_id"]==customer_id) & (t_df["transaction_id"]==transaction_id)]
    event_type=cust_transaction.iloc[0]["event_type"]
    amount=cust_transaction.iloc[0]["amount"]
    cart_value=cust_transaction.iloc[0]["cart_value"]
    timestamp=cust_transaction.iloc[0]["timestamp"]

    #Determine:NEW / RETURNING and Calculate:average_order_value
    if prev_purchases>0:
        avg_order_value=total_spending/prev_purchases
        customer_status= "Returning"
    else:
        avg_order_value=0
        customer_status= "New customer"

    #Determine:current_transaction_value
    if event_type =="PAYMENT_FAILED":
        current_transaction_value = amount
    elif event_type=="CHECKOUT_ABANDONED":
        current_transaction_value = cart_value
    elif event_type=="SUCCESSFUL_PURCHASE":
        current_transaction_value = amount

    # Determine:customer_value using CLV
    if clv < 9_000:
        customer_value="LOW"
    elif clv < 70_000:
        customer_value="MEDIUM"
    else:
        customer_value="HIGH"

    # store the results
    result["transaction_id"] = transaction_id
    result["customer_id"] = customer_id
    result["customer_status"] = customer_status
    result["prev_purchases"] = prev_purchases
    result["total_spending"] = total_spending
    result["avg_order_value"] = avg_order_value
    result["clv"] = clv
    result["current_transaction_value"] = current_transaction_value
    result["customer_value"] = customer_value

    return result
    

In [49]:
print(analyze_customer(3, 3658))

{'transaction_id': 3658, 'customer_id': 3, 'customer_status': 'Returning', 'prev_purchases': np.float64(11.0), 'total_spending': np.float64(52174.07), 'avg_order_value': np.float64(4743.097272727273), 'clv': np.float64(61098.38), 'current_transaction_value': np.float64(5830.2), 'customer_value': 'MEDIUM'}
